In [9]:
import math

# 1. Dimensions

In [10]:
# Inputs
ground_floor_area = 40
first_floor_area = 40
ground_floor_storey_height = 2.5

# RdSAP - Heights are measured internally within each room, and 0.25 m is added by software to each room height except for
# the lowest storey, to obtain the storey height.
first_floor_storey_height = ground_floor_storey_height + 0.25

# Calculations
TFA = ground_floor_area + first_floor_area
ground_V = ground_floor_area * ground_floor_storey_height
first_V = first_floor_area * first_floor_storey_height
DV =  ground_V + first_V

# Outputs
print(f"Box 4 TFA: {TFA}")
print(f"Box 5 Dwelling volume: {DV}")

Box 4 TFA: 80
Box 5 Dwelling volume: 210.0


# 2. Ventilation rate

In [11]:
# Inputs
# RdSAP - Natural with intermittent extract fans, unless mechanical system clearly identified

# Chimneys/flues/fans
open_chimneys = 0 * 80  # (6a)
open_flues = 0 * 20  # (6b)
closed_fire = 0 * 10  # (6c)
solid_fuel = 0 * 20  # (6d)
other_heater = 0 * 35  # (6e)
blocked = 0 * 20  # (6f)
fans = 0 * 10  # (7a)
vents = 0 * 10  # (7b)
flueless = 0 * 40  # (7c)
# Other inputs
has_test = False

sheltered_sides = 2 
system = 'natural'


monthly_wind = [5.10,5.00, 4.90,4.40, 4.30,3.80,3.80,3.70,4.00,4.30,4.50,4.70]

# Calculations
# Calculate infiltration from components (8) - Air changes per hour
infiltration = (open_chimneys + open_flues + closed_fire + solid_fuel + 
                other_heater + blocked + fans + vents + flueless)

infiltration_rate = infiltration / DV

if has_test:
    if ap50 > 0:
        adjusted_infiltration = (ap50 / 20) + infiltration_rate
    else:
        adjusted_infiltration = (0.263 * (ap4 ** 0.924)) + infiltration_rate
else:
    # Manual infiltration calculation
    # TODO - RdSAP - number of extract fans
    # RdSAP - 2 in other cases
    storeys = 2  
    additional = (storeys - 1) * 0.1 if storeys > 1 else 0
    
    # RdSAP - According to the largest area of wall, system build treated as masonry,
    # and infiltration according to masonry if equal. Net wall area after
    # deduction of openings is used for this purpose, walls of roof rooms are
    # not included.
    construction = 'masonry'  # Input
    structural = 0.25 if construction in ['steel', 'timber'] else 0.35
    
    # RdSAP - Age band of main dwelling A to E: unsealed
    # Age band of main dwelling F to L: sealed
    floor = 'unsealed'  # Input
    floor_infiltration = 0.2 if floor == 'unsealed' else 0.1 if floor == 'sealed' else 0
    
    # RdSAP - House, bungalow or park home: no
    lobby = 0  # Input
    lobby_infiltration = 0.05 if lobby == 0 else 0
    
    # If no draught lobby, enter 0.05, else enter 0
    proofing = 0.05  # Input
    window_infiltration = 0.25 - (0.2 * proofing / 100)
    
    adjusted_infiltration = (infiltration_rate + additional + structural + 
                            floor_infiltration + lobby_infiltration + window_infiltration)

shelter_factor = 1 - (0.075 * sheltered_sides)
final_infiltration = adjusted_infiltration * shelter_factor

# Monthly wind factor
wind_factors = [w/4 for w in monthly_wind]
adjusted_monthly = [final_infiltration * wf for wf in wind_factors]

# Mechanical ventilation
effective_ach = []
for monthly in adjusted_monthly:
    if system == 'mvhr':
        efficiency = 80
        ach = monthly + 0.5 * (1 - efficiency/100)
    elif system == 'balanced':
        ach = monthly + 0.5
    elif system == 'mechanical':
        ach = monthly + 0.5 if monthly < 0.25 else monthly + 0.25
    else:  # natural
        ach = monthly if monthly >= 1 else 0.5 + (0.5 * monthly**2)
    effective_ach.append(ach)


# Results
print(f"Box 8 Initial infiltration rate: {infiltration_rate:.2f} ACH")
print(f"Box 18 Adjusted infiltration: {adjusted_infiltration:.2f} ACH")
print(f"Box 21 With shelter factor: {final_infiltration:.2f} ACH")
print(f"Box 25 Effective ach: {[round(x, 2) for x in effective_ach]} ACH")


Box 8 Initial infiltration rate: 0.00 ACH
Box 18 Adjusted infiltration: 0.95 ACH
Box 21 With shelter factor: 0.81 ACH
Box 25 Effective ach: [1.03, 1.01, 0.99, 0.89, 0.88, 0.79, 0.79, 0.78, 0.83, 0.88, 0.91, 0.95] ACH


# 3. Heat lossses and heat loss parameter

In [12]:
# Inputs
# RdSAP - The area of an external door is taken as 1.85 m². A door to a heated access corridor is not included in the door count.
solid_door_area = 1.85

# RdSAP The U-value of insulated doors is part of the data set; the U-value of other external doors is taken from Table 15A.
solid_door_u_value = 3
doors_2 = solid_door_area * solid_door_u_value

# RdSAP -The U-value of windows and the solar transmittance of glazing is taken from Table S14.
window_inital_u = 3.1

# Table S3 : Wall thickness (mm) age band C and solid brick
wall_thickness = 0.220

ground_exposed_perimeter = 10.0     # RdSAP - Input Exposed perimeter in m
first_exposed_perimeter = 10.0

# S5.5 U-values of floors next to the ground
# Input parameters
LG = 1.5      # Thermal conductivity of clay soil (W/m·K)
RSI = 0.17    # Internal surface resistance (m²K/W)
RSE = 0.04    # External surface resistance (m²K/W)
H = 0.3       # Height above ground level (m)
V = 5.0       # Wind speed at 10m height (m/s)
FW = 0.05     # Wind shielding factor
E = 0.003     # Ventilation openings per m perimeter (m²/m)
UW = 1.5      # U-value of walls to underfloor space (W/m²K)
# Step 1: Calculate dg
dg = wall_thickness + LG * (RSI + RSE)
# Step 2: Calculate characteristic dimension B
B = 2 * ground_floor_area / ground_exposed_perimeter
# Step 3: Calculate Ug
numerator = 2 * LG * math.log(math.pi * B / dg + 1)
denominator = math.pi * B + dg
Ug = numerator / denominator
# Step 4: Calculate Ux
Ux = (2 * H * UW / B) + (1450 * E * V * FW / B)
# Step 5: Calculate final U-value
Rf = 0.2  # Uninsulated floor deck
U = 1 / (2 * RSI + Rf + 1 / (Ug + Ux))
# Round to 2 decimal places
ground_floor_u_value = round(U, 2)
ground_floor = ground_floor_area * ground_floor_u_value

# TODO - RdSAP Table S6 : Wall U-values – England and Wales
wall_u_value = 2.1

# Gross areas (inclusive of openings) are obtained from the product of heat loss perimeter (after conversion to
# internal dimensions if relevant) and storey height, summed over all storeys. 
# For the main dwelling and any extension(s), window and door areas are deducted from the gross areas to
# obtain the net wall areas for the heat loss calculations, except for the door of a flat/maisonette to an
# unheated stairwell or corridor which is deducted from the sheltered wall area.
gross_wall_area = (ground_exposed_perimeter * ground_floor_storey_height) + (first_exposed_perimeter * first_floor_storey_height)

# TODO - RdSAP - Window areas are obtained by application of the appropriate equation from Table S4
window_area  = 16.39
openings_area = solid_door_area + window_area

walls = (gross_wall_area - openings_area)*wall_u_value

# RdSAP - Roof area is the greatest of the floor areas on each level, calculated separately for main dwelling and any extension
# In the case of a pitched roof with a sloping ceiling, divide the area so obtained by cos(30°).
roof_area = ground_floor_area
# TODO - RdSAP Table S10 : Assumed roof U-values when Table S9 does not apply
roof_u_value = 2.3
roof = roof_area * roof_u_value

# RdSAP - The thermal mass parameter is taken as 250 kJ/m²K.
TMP = 250
days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]


linear_thermal_bridges = 0
point_thermal_bridges = 0


# TODO - RdSAP The U-value of party walls is taken from Table S8B
party_wall_u_value =0.25

# RdSAP - Party wall area is party wall length multiplied by storey height, summed over all storeys
party_wall_length_ground = 10
party_wall_length_first = 10
party_wall_area = (party_wall_length_ground * ground_floor_storey_height) + (party_wall_length_first * first_floor_storey_height)

# Calcuations
u_value_windows = 1/((1/window_inital_u)+0.04)

windows = window_area * u_value_windows

total_area_of_external_elements = solid_door_area + window_area + roof_area + (gross_wall_area-openings_area) + ground_floor_area 

party_wall_heat_loss = party_wall_area * party_wall_u_value

fabric_heat_loss = doors_2 +windows + ground_floor + walls +roof +party_wall_heat_loss

thermal_bridges = 0.15 * total_area_of_external_elements
total_fabric_heat_loss = fabric_heat_loss + thermal_bridges
ventilation_heat_loss = [0.33 * ach * DV for ach in effective_ach]
HTC = [vent + total_fabric_heat_loss for vent in ventilation_heat_loss ]
HTC_avg = sum(HTC) / len(HTC)
HLP = [HT / TFA for HT in HTC ]

# Results
print(f"Box 31 Total area of external elements {total_area_of_external_elements:.2f}")
print(f"Box 33 Fabric heat loss {fabric_heat_loss:.2f}")
print(f"Box 37 Total fabric heat loss {total_fabric_heat_loss:.2f}")
print(f"Box 38 Ventilation heat loss {[round(x, 2) for x in ventilation_heat_loss]}")
print(f"Box 39 HTC {[round(x, 2) for x in HTC]}")
print(f"Box 40 HLP {[round(x, 2) for x in HLP]}")

Box 31 Total area of external elements 132.50
Box 33 Fabric heat loss 248.22
Box 37 Total fabric heat loss 268.10
Box 38 Ventilation heat loss [71.34, 69.94, 68.55, 61.98, 60.75, 55.04, 55.04, 53.98, 57.24, 60.75, 63.24, 65.84]
Box 39 HTC [339.44, 338.04, 336.65, 330.08, 328.85, 323.14, 323.14, 322.08, 325.34, 328.85, 331.34, 333.94]
Box 40 HLP [4.24, 4.23, 4.21, 4.13, 4.11, 4.04, 4.04, 4.03, 4.07, 4.11, 4.14, 4.17]
